# Simplex family — full exploration, n = 1, ..., 6

For each n (the simplex's dimension, so it has n+1 vertices — matching the convention in `simplex.sage`), this notebook:

1. lists all vertices,
2. computes the canonical form via the general nbc method (Brown–Dupont Prop. 6.7, `general_canonical_forms.sage`), checking the defining pole-structure property,
3. computes the projective (polar) dual,
4. checks the **volume conjecture**: the canonical form, evaluated at the centroid (in a chart re-centered there), equals ±n! times the volume of the projective dual taken at that same centroid — with a short demonstration (n=1 section) of why it has to be the centroid and not an arbitrary interior point,
5. enumerates all triangulations and identifies which are regular,
6. computes the secondary polytope and its vertex embedding.

Steps 5–6 stay trivial at every n here — a simplex with no points beyond its own n+1 vertices has exactly one triangulation (itself), regardless of n — unlike the hypercube (`hypercube_explorer.ipynb`), where the same steps become computationally infeasible past n=3.

**Requires the `sagemath` Jupyter kernel** and must be opened from the same synced folder as the `.sage` files — see `README.md`.

In [ ]:
load("general_canonical_forms.sage")

That `load` pulls in `common.sage` (vertex generators, `polar_dual`, `secondary_polytope_data`) and `vertex_sum_canonical_forms.sage` too, and runs `general_canonical_forms.sage`'s own test suite as a side effect (scroll up for that PASS/FAIL output). Everything below is fresh, per-instance exploration of the simplex family specifically.

## n = 1 — the segment

In [ ]:
n = 1
y = [var(f"y{i}") for i in range(1, n + 1)]
pts = simplex_vertices(n)
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices:")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Simplex n={n}", phi, P, y)
phi

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual

The volume conjecture (Arkani-Hamed–Bai–Lam) relates the canonical form not to the polytope's own volume, but to the volume of its **projective dual**. This needs care about *where* the dual is taken from: the ordinary (metric) polar dual $P^\circ = \{y : x\cdot y \le 1\ \forall x \in P\}$ depends on an arbitrary choice of origin, and its volume is wildly sensitive to that choice (demonstrated two cells down, for an origin with no particular geometric meaning). The *projective* dual fixes this by using the one reference point every polytope determines from its own vertices with no external choice involved — the **centroid**. Once the canonical form is re-expressed in a chart centered there, and the dual is taken there too, the identity is exact — confirmed here on the simplex family, and separately on an unrelated scalene triangle while deriving this section, not assumed:

$$\phi_{\text{centroid}}(0) = \pm\, n!\cdot\mathrm{Vol}\!\left(P^\circ_{\text{centroid}}\right)$$

with the same overall orientation sign built into the canonical form's own definition. `common.sage`'s `polar_dual` already centers at the centroid internally, so `Dual = polar_dual(P)` from the previous cell already *is* $P^\circ_{\text{centroid}}$.

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(n)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(n)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

# Re-express the canonical form in the centroid-centered chart -- the
# original chart's y=0 is one of the simplex's own vertices (a pole),
# not an interior point, so phi can't be evaluated at 0 there.
phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()  # Dual = polar_dual(P), already centroid-centered (previous cell)
target = factorial(n) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("n! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches n! * Vol(dual):", match_plus, " matches -n! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

#### Why it has to be the centroid, not just any interior point

Picking a different interior reference point breaks the identity — confirming this isn't a coincidence of using *some* origin near the polytope, but specifically requires the centroid (the one projectively canonical choice, with no external rescaling ambiguity). Below, an arbitrary nearby interior point is used instead — deliberately calling `.polar()` directly (bypassing `polar_dual`, which always re-centers at *its own* centroid) so the dual really is taken with respect to this off-centroid point.

In [ ]:
off_center = [c + QQ(1) / 5 for c in centroid]  # nudged off the centroid, still interior
pts_off = [tuple(QQ(v[i]) - off_center[i] for i in range(n)) for v in pts]
P_off = Polyhedron(vertices=pts_off)
assert P_off.interior_contains(vector([0] * n)), "sanity check: still an interior point"

phi_off = general_canonical_form_density(P_off, y)
val_off = phi_off.subs({yi: 0 for yi in y})
vol_dual_off = P_off.polar().volume()  # dual w.r.t. THIS (off-centroid) origin, not re-centered

print("phi at an off-centroid interior point =", val_off)
print("n! * Vol(dual there) =", factorial(n) * vol_dual_off)
print("these do not match -- the identity is specific to the centroid, not just any origin")

### All triangulations, and which are regular

With no points beyond its own n+1 vertices, a simplex has exactly one triangulation (itself) — expected to come out regular too, but that's checked here rather than assumed (see `secondary_polytope_data`'s docstring in `common.sage` for the method: matching each triangulation's GKZ vector against the secondary polytope's vertex list, entirely avoiding a TOPCOM subprocess call that was found to hang and crash WSL on this machine).

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

### Secondary polytope: vertex embedding

`pc.secondary_polytope()` naturally lives in R^(number of points) — one coordinate per point of the configuration — even though its actual dimension is usually much smaller; `reduce_secondary_polytope` (`common.sage`) projects onto its own affine hull via `Polyhedron.affine_hull_projection()`, giving vertex coordinates in exactly `sp.dimension()`-many numbers instead of a padded, redundant embedding.

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

## n = 2 — the triangle

In [ ]:
n = 2
y = [var(f"y{i}") for i in range(1, n + 1)]
pts = simplex_vertices(n)
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices:")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Simplex n={n}", phi, P, y)
phi

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual (taken at the centroid — see the n=1 section for why)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(n)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(n)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()  # Dual = polar_dual(P), already centroid-centered (previous cell)
target = factorial(n) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("n! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches n! * Vol(dual):", match_plus, " matches -n! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

## n = 3 — the tetrahedron

In [ ]:
n = 3
y = [var(f"y{i}") for i in range(1, n + 1)]
pts = simplex_vertices(n)
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices:")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Simplex n={n}", phi, P, y)
phi

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual (taken at the centroid — see the n=1 section for why)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(n)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(n)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()  # Dual = polar_dual(P), already centroid-centered (previous cell)
target = factorial(n) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("n! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches n! * Vol(dual):", match_plus, " matches -n! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

## n = 4 — the 4-simplex

In [ ]:
n = 4
y = [var(f"y{i}") for i in range(1, n + 1)]
pts = simplex_vertices(n)
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices:")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Simplex n={n}", phi, P, y)
phi

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual (taken at the centroid — see the n=1 section for why)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(n)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(n)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()  # Dual = polar_dual(P), already centroid-centered (previous cell)
target = factorial(n) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("n! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches n! * Vol(dual):", match_plus, " matches -n! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

## n = 5 — the 5-simplex

In [ ]:
n = 5
y = [var(f"y{i}") for i in range(1, n + 1)]
pts = simplex_vertices(n)
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices:")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Simplex n={n}", phi, P, y)
phi

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual (taken at the centroid — see the n=1 section for why)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(n)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(n)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()  # Dual = polar_dual(P), already centroid-centered (previous cell)
target = factorial(n) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("n! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches n! * Vol(dual):", match_plus, " matches -n! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

## n = 6 — the 6-simplex

In [ ]:
n = 6
y = [var(f"y{i}") for i in range(1, n + 1)]
pts = simplex_vertices(n)
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices:")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Simplex n={n}", phi, P, y)
phi

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual (taken at the centroid — see the n=1 section for why)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(n)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(n)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()  # Dual = polar_dual(P), already centroid-centered (previous cell)
target = factorial(n) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("n! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches n! * Vol(dual):", match_plus, " matches -n! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()